# Data Science: Feature Selection via Continuous Relaxation

Feature-subset selection is naturally binary (include/exclude each feature), but instead of writing a separate binary-encoded algorithm we reuse the continuous optimizers: each dimension is a weight $w_i \in [0, 1]$, thresholded at $0.5$ to get the subset mask inside the objective function. This is the same trick behind 'sigmoid'/continuous-relaxation binary PSO in the literature, and it means any of `GeneticAlgorithm`, `ParticleSwarmOptimization`, `DifferentialEvolution`, or `SlimeMouldAlgorithm` can be used unmodified.

Objective: cross-validated classification error, plus a small penalty on the number of features used (so the optimizer doesn't just keep everything):

$$f(w) = \left(1 - \text{CV accuracy on features where } w_i > 0.5\right) + \alpha \cdot \frac{|\{i : w_i > 0.5\}|}{N}$$

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

from metaheuristics.algorithms.particle_swarm import ParticleSwarmOptimization

X, y = load_breast_cancer(return_X_y=True)
NUM_FEATURES = X.shape[1]
ALPHA = 0.02  # weight on the feature-count penalty

In [3]:
def feature_subset_error(weights):
    mask = weights > 0.5
    if not mask.any():
        return 1.0
    score = cross_val_score(SVC(), X[:, mask], y, cv=5).mean()
    return (1 - score) + ALPHA * mask.sum() / NUM_FEATURES

bounds = [(0.0, 1.0)] * NUM_FEATURES

np.random.seed(0)
result = ParticleSwarmOptimization(num_particles=15, max_iterations=15).optimize(feature_subset_error, bounds)
mask = result.best_solution > 0.5

baseline_accuracy = cross_val_score(SVC(), X, y, cv=5).mean()
selected_accuracy = cross_val_score(SVC(), X[:, mask], y, cv=5).mean()

print(f'Selected {mask.sum()}/{NUM_FEATURES} features')
print(f'CV accuracy, all features     : {baseline_accuracy:.4f}')
print(f'CV accuracy, selected features: {selected_accuracy:.4f}')

Selected 9/30 features
CV accuracy, all features     : 0.9122
CV accuracy, selected features: 0.9385
